## Self-attention

The __self-attention__ computation is at the core of the Transformer architecture. The input to the self-attention computation is a tensor containing a vector $x$ for each input token, and the output is a tensor $y$ of the same dimensions, containing the contextualized versions of the input tokens. 

Each such vector $x$ projects three vectors: a __query__ $q$, a __key__ $k$, and a __value__ $v$. We first proceed to compute the dot product of every pair of query vector $q_i$ and key vector $k_j$. For instance, if we have 4 input tokens, we get the following situation:

![Self-attention](img/attention1.png)

In general, when two vectors have positive and negative values distributed the same way, they tend to have a higher dot product. As a simple example, if $q_1 = (1.2, -0.8, -0.6)$ and $k_2 = (0.8, -1.1, -0.2)$, then $q_1\cdot k_2 = 1.96$, but if $k_2 = (-0.8, -1.1, -0.2)$, then $q_1\cdot k_2$ is only $0.04$, due to the different signs in the first dimension. If the vectors have different signs in many dimensions, the dot product can be large negative. 

If we think of the dimensions as the axes in a semantic vector space, a large dot product is a sign that the query and key ``match''. You might think of the query $q_1$ as specifying what that the first token is looking for along those dimensions, and the key $k_2$ as specifying what the second token is offering. A large dot product $q_1\cdot k_2$ indicates that the second token is highly relevant to incorporate in the contextualized version of the first token. 

In order to get less extreme numbers, we divide the dot products by the square root of the vector dimension ($\sqrt{3}$ in our simple example, but in real applications the vectors would have several hundred dimensions). 

If we then softmax all these dot products along the rows of the matrix above, we get weights adding up to 1, for instance:

![Softmaxxed self-attention](img/attention2.png)

The first row now specifies that the contextualized version of the first token should be a weighted sum (20\% of the first token, 20\% of the second token, etc.). More specifically, we compute the weighted sum of the __values__, e.g., $y_1 = 0.2v_1 + 0.2v_2 + 0.55v_3 + 0.05v_4$. So, the value vectors $v_j$ are what each token is contributing to the final representation.

For generative applications, for instance a language model where we want to generate words conditioned on some starting words, we don't want future tokens to influence preceding tokens, as those future tokens would not be available at inference time. In this case, we apply a so-called __causal mask__ to the dot-product matrix:

![Causal mask](img/attention3.png)

For each masked cell (with a 0 value), we replace the dot product in the corresponding cell by $-\infty$:

![Masked attention](img/attention4.png)

When applying softmax to a row in the matrix, we then ensure that these masked cells don't contribute to the final contextualized vector (as $e^{-\infty} = 0$).

Your task is to fill in the missing pieces below. Look for "REPLACE WITH YOUR CODE" and "YOUR CODE HERE". Note that the flag ``is_causal`` should control whether or not you should apply a mask as illustrated above.


In [ ]:
# First run this cell
import torch
from torch import nn
import math

In [ ]:
class SelfAttention(nn.Module):
    """
    Computes self-attention according to Vashwani et al, 2017.
    """

    def __init__(self, vector_dim, block_size, is_causal=False):
        """
        vector_dim = the dimension of the input and output vectors
        block_size = max number of tokens in the input 
        is_causal is True if tokens only can be influenced by preceding
                  tokens (typical in generative applications)
        """
        super().__init__()
        self.vector_dim = vector_dim
        self.is_causal = is_causal
        self.wq = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wk = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wv = nn.Linear(vector_dim, vector_dim, bias=False)
        self.wo = nn.Linear(vector_dim, vector_dim, bias=False)
        if self.is_causal:
            # The 'causal mask' is a lower-left triangular matrix of 1s, wrapped in
            # the outermost dimension(s) (which is the batch and, in the multihead
            # case, the number_of_heads dimension)
            causal_mask = torch.tril(torch.ones(block_size, block_size)).unsqueeze(0)
            # The next line creates a buffer 'self.mask'. Using a buffer rather than
            # a parameter ensures it will be moved to the GPU along with parameters,
            # but it won't be changed during training.
            self.register_buffer("mask", causal_mask)

    def compute_attention(self, q, k, v):
        # Shape of the tensors are (B,S,D) = (batch size, seq length, attention dim)
        # In the single-head case, attention_dim = vector_dim

        # YOUR CODE HERE
        
        if self.is_causal:
            pass  # REPLACE WITH YOUR CODE 

        # YOUR CODE HERE
    
        return values

    def forward(self, x):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        values = self.compute_attention(q, k, v)
        out = self.wo(values)
        return out


In [ ]:
seed = 4224
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 1
block_size = 32
seq_len = 20 
vector_dim = 64
attention = SelfAttention(vector_dim, block_size, is_causal=False)
data = torch.rand(batch_size, seq_len, vector_dim)
result = attention(data)
print( "Without causal mask" )
print(f'{result[0][7][7].detach().item():.4f}' == '0.0342')
print(f'{result[0][8][8].detach().item():.4f}' == '0.2462')
print(f'{result[0][9][9].detach().item():.4f}' == '-0.1948')
print(f'{result[0][10][10].detach().item():.4f}' == '0.0360')
attention = SelfAttention(vector_dim, block_size, is_causal=True)
result = attention(data)
print( "With causal mask" )
print(f'{result[0][7][7].detach().item():.4f}' == '0.0154')
print(f'{result[0][8][8].detach().item():.4f}' == '-0.2425')
print(f'{result[0][9][9].detach().item():.4f}' == '0.3206')
print(f'{result[0][10][10].detach().item():.4f}' == '0.2764')

### Multi-head self-attention
It has proven to be very useful to split up the query, key and value matrices in several parts and perform the self-attention computation separately on each part. Each such part is called a __head__. Ignoring the batch dimension for a moment, consider the $q$ matrix below, where we have $s$ rows corresponding to $s$ input tokens, and the dimensionality of the vectors is 6:

![q matrix](img/attention5.png)

We now want to divide the self-attention computation into 3 heads. The number of heads must be a divisor of the vector dimension -- 6 in this case -- so $6//3=2$ becomes the "attention dimensionality". Therefore, we split the matrix as follows:

![q matrix with parts](img/attention5.png)

and rearrange the original $s\times 6$ matrix into three $s\times 2$ matrices (or rather, a ``[3,s,2]`` tensor):

![q tensor](img/attention5.png)

Due to broadcasting, the old self-attention code will still work as intended, even though we now have an extra dimension compared to the single-head case. After the self-attention computation, the resulting tensor should be reshaped to the original shape (which would be $s\times 6$ in the example). 

Your task is now to implement these reshaping operations below.




In [ ]:
class MultiHeadSelfAttention(SelfAttention):
    """
    Computes self-attention with multiple heads according to Vashwani et al, 2017.
    """

    def __init__(self, vector_dim, n_heads, block_size, is_causal=False):
        """
        vector_dim = the dimension of the input and output vectors
        n_heads = the number_of_attention_heads. It has to be a divisor of vector_dim
        block_size = max number of tokens in the input 
        is_causal is True if tokens only can be influenced by preceding
                  tokens (typical in generative applications)
        """   
        super().__init__(vector_dim, block_size, is_causal)
        self.att_dim = vector_dim//n_heads
        self.n_heads = n_heads

    def reshape_for_multihead_attention(self, x):
        """
        x has the shape (batch_size, seq_length, vector_dim)

        We want to split the representation of each token into 'number_of_heads'
        parts and treat each part separately. Thus, we need the returned tensor
        to have shape (batch_size, no_of_heads, seq_length, att_dim)
        """

        # YOUR CODE HERE

        return None # REPLACE WITH YOUR CODE

    def reshape_after_multihead_attention(self, x):
        """
        x has the shape (batch_size, no_of_heads, seq_length, att_dim)

        For each token, we now want to bring together the representation coming
        from each head. The returned token should have the shape:
        (batch_size, seq_length, vector_dim)
        """

        # YOUR CODE HERE

        return None # REPLACE WITH YOUR CODE

    def forward(self, x):
        q = self.wq(x)
        k = self.wk(x)
        v = self.wv(x)
        q = self.reshape_for_multihead_attention(q)
        k = self.reshape_for_multihead_attention(k)
        v = self.reshape_for_multihead_attention(v)
        values = self.compute_attention(q, k, v)
        values = self.reshape_after_multihead_attention(values)
        out = self.wo(values)
        return out

In [ ]:
"""
Test the MultiHeadSelfAttention class. Make sure that you first run the cell above!
"""
seed = 4224
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 1
block_size = 32
seq_len = 20
vector_dim = 64
no_of_heads = 2

attention = MultiHeadSelfAttention(vector_dim, no_of_heads, block_size, is_causal=False)
data = torch.rand(batch_size, seq_len, vector_dim)

print("Check that reshaping works:")
desired_shape = torch.Size([batch_size, attention.n_heads, seq_len, att_dim])
reshaped_data = attention.reshape_for_multihead_attention(data)
print( reshaped_data.shape == desired_shape )
print( (data[0][9][33] == reshaped_data[0][1][9][1]).detach().item() )
rereshaped_data = attention.reshape_after_multihead_attention(reshaped_data)
print( data.shape == rereshaped_data.shape )
print( torch.all(data == rereshaped_data).detach().item() )

result = attention(data)
print ("Multihead attention without causal mask")
print(f'{result[0][7][7].detach().item():.4f}' == '0.0336')
print(f'{result[0][8][8].detach().item():.4f}'== '0.2456')
print(f'{result[0][9][9].detach().item():.4f}' == '-0.1955')
print(f'{result[0][10][10].detach().item():.4f}' == '0.0356')

attention = MultiHeadSelfAttention(vector_dim, no_of_heads, block_size, is_causal=True)
result = attention(data)
print ("Multihead attention with causal mask")
print(f'{result[0][7][7].detach().item():.4f}' == '0.0147')
print(f'{result[0][8][8].detach().item():.4f}'== '-0.2433')
print(f'{result[0][9][9].detach().item():.4f}' == '0.3209')
print(f'{result[0][10][10].detach().item():.4f}' == '0.2763')